In [ ]:
import base64, gzip, json, os, sys
from pathlib import Path
bundle = json.loads(gzip.decompress(base64.b64decode("...")))
root = Path("/kaggle/working/ieee_cis_fraud_code/src/fraudlab")
root.mkdir(parents=True, exist_ok=True)
for name, source in bundle.items():
    (root / name).write_text(source, encoding="utf-8")
os.environ["FRAUDLAB_SRC"] = str(root.parent)
sys.path.insert(0, str(root.parent))
print("Project modules installed:", len(bundle))

# Triple A IEEE-CIS 03C — Immediate Feedback Oracle\n\nSensitivity experiment only: this simulation period was already analyzed in the historical run, so these results describe assumption sensitivity and must not be used to select a winner. Fixed LightGBM UID scores, learner settings, transaction stream, costs and 3% daily review cap. Three seeds (42, 43, 44).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from fraudlab.config import ExperimentConfig, FeedbackConfig
from fraudlab.metrics import classification_metrics, REVIEW
from fraudlab.pipeline import load_simulation_input
from fraudlab.paths import resolve_artifact_dir
from fraudlab.simulator import run_bandit_simulation, run_threshold_baselines, save_simulation_outputs

BRANCH = 'immediate_feedback_oracle'
BLOCKED_DELAY_DAYS = 0
SEEDS = (42, 43, 44)
OUTPUT_DIR = Path('/kaggle/working/artifacts/simulation') / BRANCH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_ROOT = resolve_artifact_dir('features/ieee_cis_features.parquet')
EXPERIMENT_ROOT = resolve_artifact_dir('experiments/selected_feature_recipe.json')
EXPERIMENT_DIR = EXPERIMENT_ROOT / 'experiments'
SELECTED_RECIPE = json.loads((EXPERIMENT_DIR / 'selected_feature_recipe.json').read_text())['recipe']
config = ExperimentConfig()
RECIPE_DIR = EXPERIMENT_DIR / 'recipes' / SELECTED_RECIPE
prediction_path = RECIPE_DIR / 'predictions' / 'lightgbm_simulation_predictions.parquet'
rows, probabilities = load_simulation_input(
    FEATURE_ROOT / 'features' / 'ieee_cis_features.parquet', prediction_path, config
)
threshold_config = json.loads((RECIPE_DIR / 'lightgbm_thresholds.json').read_text())
baselines = run_threshold_baselines(rows, probabilities, config.costs, threshold_config)
diagnostic_metrics = classification_metrics(rows['isFraud'].to_numpy(), probabilities)
print(f'Branch: {BRANCH} | rows: {len(rows)} | recipe: {SELECTED_RECIPE} | seeds: {SEEDS}')

In [ ]:
from collections import defaultdict
import numpy as np

metric_rows = []
paired_costs = {}
for seed in tqdm(SEEDS, desc=f'{BRANCH}: seed progress', unit='seed'):
    feedback_cfg = FeedbackConfig(
        review_delay_days=1, maturity_days=30, fraud_delay_min_days=1,
        fraud_delay_max_days=30, fraud_censor_probability=0.20,
        blocked_label_delay_days=BLOCKED_DELAY_DAYS, seed=seed,
    )
    modes = ('oracle',) if BRANCH == 'immediate_feedback_oracle' else ('closed_loop', 'oracle')
    run_results = {}
    for mode in modes:
        decisions, metrics = run_bandit_simulation(
            rows, probabilities, mode, config.costs, feedback_cfg, config.bandit, seed=seed
        )
        metrics.update({'model':'lightgbm','scenario':BRANCH,'seed':seed,'experiment':mode})
        daily = decisions.groupby('day').action_id
        daily_reviews = daily.apply(lambda x: int((x == REVIEW).sum()))
        daily_caps = daily.size().apply(lambda n: int(np.floor(config.costs.review_cap*n)))
        assert (daily_reviews <= daily_caps).all(), f'{mode}: daily review cap exceeded'
        if mode == 'closed_loop':
            if BRANCH == 'blocked_labels_hidden':
                assert decisions.loc[decisions.action_id == 2, 'label_release_day'].isna().all()
                assert metrics['blocked_labels_released_by_horizon'] == 0
            elif BRANCH == 'blocked_labels_after_30_days':
                block_rows = decisions.action_id == 2
                assert (decisions.loc[block_rows, 'evaluator_feedback_scheduled_day'] ==
                        decisions.loc[block_rows, 'day'] + 30).all()
                assert metrics['blocked_labels_permanently_censored'] == 0
        if mode == 'oracle':
            assert (decisions['label_release_day'] == decisions['day']).all()
            assert metrics['feedback_updates'] == len(decisions)
        save_simulation_outputs(
            decisions, metrics, OUTPUT_DIR,
            f'lightgbm_{BRANCH}_seed{seed}_{mode}',
        )
        metric_rows.append({
            'record_type':'simulation','model':'lightgbm','scenario':BRANCH,'seed':seed,
            'experiment':mode, **metrics, **diagnostic_metrics,
        })
        run_results[mode] = metrics
    if 'closed_loop' in run_results:
        tax = run_results['closed_loop']['total_cost'] - run_results['oracle']['total_cost']
        paired_costs[seed] = tax
        for row in metric_rows[-2:]:
            row['oracle_tax'] = tax
    else:
        metric_rows[-1]['oracle_tax'] = 0.0

for baseline, values in baselines.items():
    metric_rows.append({
        'record_type':'baseline','model':'lightgbm','scenario':'common_baseline',
        'seed':np.nan,'experiment':baseline,'oracle_tax':np.nan,**values,**diagnostic_metrics,
    })

metrics_frame = pd.DataFrame(metric_rows)
metrics_frame.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
simulation_rows = metrics_frame.loc[metrics_frame.record_type == 'simulation']
metric_columns = [
    'total_cost','oracle_tax','approve_count','review_count','block_count',
    'review_rate','review_cap_utilization','feedback_updates',
    'feedback_events_pending_at_end','blocked_labels_scheduled',
    'blocked_labels_released_by_horizon','blocked_labels_pending_at_end',
    'blocked_labels_permanently_censored','approved_fraud_labels_permanently_censored',
    'pr_auc','precision_at_0_5','recall_at_0_5',
]
summary = simulation_rows.groupby('experiment')[metric_columns].agg(['mean','std'])
summary.to_csv(OUTPUT_DIR / 'metrics_summary.csv')
print('Per-seed results:')
print(simulation_rows[['seed','experiment','total_cost','oracle_tax','approve_count','review_count','block_count',
                       'feedback_updates','blocked_labels_scheduled','blocked_labels_released_by_horizon',
                       'blocked_labels_pending_at_end','blocked_labels_permanently_censored',
                       'pr_auc','precision_at_0_5','recall_at_0_5']].to_string(index=False))
print('\nMean and standard deviation across seeds:')
print(summary.to_string())
print('\nBaselines:')
print(metrics_frame.loc[metrics_frame.record_type == 'baseline',
                        ['experiment','total_cost','review_count','pr_auc','precision_at_0_5','recall_at_0_5']].to_string(index=False))
print('\nOutputs:', OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

sim = metrics_frame.loc[metrics_frame.record_type == 'simulation']
grouped = sim.groupby('experiment').total_cost.agg(['mean','std'])
fig, ax = plt.subplots(figsize=(8, 5))
labels = list(grouped.index)
ax.bar(labels, grouped['mean'], yerr=grouped['std'].fillna(0), capsize=4, color=['#3977a8','#7a9e45'][:len(labels)])
base = metrics_frame.loc[metrics_frame.record_type == 'baseline'].set_index('experiment').total_cost
for name, color in [('no_review','#d17a22'),('validation_tuned_thresholds','#a34444')]:
    if name in base.index:
        ax.axhline(base[name], color=color, linestyle='--', label=f'{name}: {base[name]:,.0f}')
ax.set_title(f'LightGBM bandit feedback sensitivity — {BRANCH}\\nExploratory reuse of previously analyzed simulation period')
ax.set_ylabel('Total decision cost (points)')
ax.set_xlabel('Feedback mode')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'cost_comparison.png', dpi=150)
plt.show()